# 03 — New Model (ConvNeXt-Base)

Benchmarks a new architecture (ConvNeXt-Base) against the paper baseline on the
ceilometer backscatter dataset, with training tracked on **Weights & Biases**.
Logged to the same `cloud-detection` project as `02_baseline.ipynb` for direct comparison.

Runs both locally and on Google Colab — the dataset path comes from
`configs/config.yaml` (`dataset.path`), so update that file to point at wherever
the dataset actually lives (a local folder or Google Drive).

In [ ]:
import sys, os

if os.path.exists('/content'):
    # Siamo su Colab
    repo_path = '/content/Cloud_detection_project'
    if not os.path.exists(repo_path):
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
        os.system(f'git clone https://{token}@github.com/caMatt99/Cloud_detection_project.git {repo_path}')
    sys.path.insert(0, f'{repo_path}/src')
    os.chdir(repo_path)
else:
    # Siamo in locale — il notebook gira da notebooks/, torniamo alla root del progetto
    os.chdir('..')
    sys.path.insert(0, 'src')

In [ ]:
import yaml
import torch.nn as nn
import torch.optim as optim
import wandb

from dataset import get_dataloaders
from models import get_model, get_device, count_parameters
from evaluate import get_predictions, compute_metrics
from engine import train_one_epoch, evaluate_one_epoch, log_epoch_to_wandb

In [ ]:
if os.path.exists('/content'):
    # Siamo su Colab — chiave letta dal Secret manager (icona a chiave nella sidebar)
    from google.colab import userdata
    wandb.login(key=userdata.get('WANDB_API_KEY'))
else:
    # Siamo in locale — usa le credenziali salvate in ~/.netrc (wandb login da terminale)
    wandb.login()

In [ ]:
config_path = os.path.join(os.getcwd(), 'configs', 'config.yaml')
with open(config_path) as f:
    cfg = yaml.safe_load(f)

env = "colab" if os.path.exists('/content') else "local"

config = {
    "model_name":    cfg["model"]["name"],  # "convnext_base"
    "dataset_path":  cfg["dataset"]["path"][env],
    "batch_size":    cfg["dataset"]["batch_size"],
    "image_size":    cfg["dataset"]["image_size"],
    "num_workers":   2,
    "num_classes":   cfg["model"]["num_classes"],
    "epochs":        cfg["training"]["epochs"],
    "optimizer":     cfg["training"]["optimizer"],
    "lr":            cfg["training"]["learning_rate"],
    "momentum":      cfg["training"]["momentum"],
    "weight_decay":  cfg["training"]["weight_decay"],
}

In [ ]:
run = wandb.init(
    project="cloud-detection",
    name=f"new-model-{config['model_name']}",
    job_type="train",
    config=config,
)
config = wandb.config

In [ ]:
device = get_device()

train_loader, val_loader, test_loader, class_names = get_dataloaders(
    dataset_path=config["dataset_path"],
    batch_size=config["batch_size"],
    image_size=config["image_size"],
    num_workers=config["num_workers"],
)

model = get_model(
    config["model_name"], num_classes=config["num_classes"], pretrained=True
).to(device)
count_parameters(model)

run.watch(model, log="all", log_freq=10)

In [ ]:
criterion = nn.CrossEntropyLoss()

if config["optimizer"] == "sgd":
    optimizer = optim.SGD(
        model.parameters(),
        lr=config["lr"],
        momentum=config["momentum"],
        weight_decay=config["weight_decay"],
    )
else:
    optimizer = optim.Adam(
        model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"]
    )

## Training loop

`train_one_epoch` / `evaluate_one_epoch` / `log_epoch_to_wandb` live in `src/engine.py`
(shared with `02_baseline.ipynb`). Each epoch logs `train/loss`, `train/accuracy`,
`val/loss`, `val/accuracy`, and the validation confusion matrix to W&B.

In [ ]:
for epoch in range(1, config["epochs"] + 1):
    train_loss, train_acc, _, _ = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    val_loss, val_acc, val_preds, val_labels = evaluate_one_epoch(
        model, val_loader, criterion, device
    )

    log_epoch_to_wandb(
        epoch, train_loss, train_acc, val_loss, val_acc, val_preds, val_labels, class_names
    )

    print(
        f"Epoch {epoch:3d}/{config['epochs']}  "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f}  "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
    )

## Final evaluation on the test set

In [ ]:
test_preds, test_labels = get_predictions(model, test_loader, device)
test_metrics = compute_metrics(test_preds, test_labels, model_name=config["model_name"])

wandb.log(
    {
        "test/accuracy": test_metrics["accuracy"],
        "test/f1": test_metrics["f1"],
        "test/precision": test_metrics["precision"],
        "test/recall": test_metrics["recall"],
        "test/confusion_matrix": wandb.plot.confusion_matrix(
            preds=test_preds,
            y_true=test_labels,
            class_names=class_names,
        ),
    }
)

## Save checkpoint and log it as a W&B artifact

In [ ]:
import torch

checkpoint_dir = os.path.join(os.getcwd(), "checkpoints")
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_path = os.path.join(checkpoint_dir, f"{config['model_name']}.pt")
torch.save(model.state_dict(), checkpoint_path)

artifact = wandb.Artifact(
    name=f"{config['model_name']}-checkpoint",
    type="model",
    metadata=dict(test_metrics),
)
artifact.add_file(checkpoint_path)
run.log_artifact(artifact)

run.finish()